# Day 3—Turning text into data
*Measuring Manuscripts*

Before a computer can count words, you have to define a word, and the definition shapes everything downstream. Today we cut running text into units—tokens, types, lemmas—and build the two oldest tools in the field: a frequency list and a concordance.

## 1. Load a text

A short public-domain fable is built in so everything runs. Replace it with your own corpus once the pieces are clear.

In [ ]:
# === TO BUILD: load a real corpus (e.g. a Coptic SCRIPTORIUM text) in place of `sample` ===
CORPUS_URL = None

sample = (
    "The North Wind and the Sun were disputing which was the stronger, when a traveler came along wrapped in a warm cloak. They agreed that the one who first succeeded in making the traveler take his cloak off should be considered stronger than the other. Then the North Wind blew as hard as he could, but the more he blew the more closely did the traveler fold his cloak around him; and at last the North Wind gave up the attempt. Then the Sun shone out warmly, and immediately the traveler took off his cloak. And so the North Wind was obliged to confess that the Sun was the stronger of the two."
)
text = sample
print(text[:120], '...')

## 2. Tokenizing is already a decision

Splitting on spaces is the obvious move, but it carries small choices. Is *cloak.* the same token as *cloak*? Is *Sun* the same as *sun*? Watch the count change as we normalize.

In [ ]:
naive = text.split()
clean = [w.lower().strip('.,;:!?') for w in text.split()]

print('Naive split:        ', len(naive), 'tokens,', len(set(naive)), 'types')
print('Lowercased, stripped:', len(clean), 'tokens,', len(set(clean)), 'types')
print('\nWe lost', len(set(naive)) - len(set(clean)), 'apparent types just by cleaning up.')

## 3. Types, tokens, and the rare words

A **token** is each running word; a **type** is each distinct form. Words that occur exactly once are *hapax legomena*, and there are always more of them than you'd guess.

In [ ]:
from collections import Counter
tokens = clean
freq = Counter(tokens)
hapax = [w for w, n in freq.items() if n == 1]

print('Tokens:', len(tokens))
print('Types: ', len(freq))
print('Hapax (appear once):', len(hapax), '->', hapax[:8], '...')

## 4. Frequency list, with a picture

In [ ]:
import matplotlib.pyplot as plt
top = freq.most_common(12)
for word, n in top:
    print(f'{word:10} {n}')

labels = [w for w, _ in top]
counts = [n for _, n in top]
plt.figure(figsize=(8, 3))
plt.bar(labels, counts)
plt.xticks(rotation=45, ha='right'); plt.title('Most frequent words'); plt.tight_layout(); plt.show()

The most frequent words are almost always grammatical—*the*, *and*, *of*. The informative words begin further down the list. Day 6 measures how this distribution behaves.

## 5. Keyword in context (KWIC)

A concordance shows every occurrence of a word with its neighbors. Regularities invisible in running text become obvious once the occurrences are aligned.

In [ ]:
def kwic(tokens, keyword, width=4):
    hits = 0
    for k, w in enumerate(tokens):
        if w == keyword:
            left  = ' '.join(tokens[max(0, k - width):k])
            right = ' '.join(tokens[k + 1:k + 1 + width])
            print(f'{left:>28}  [ {keyword} ]  {right}')
            hits += 1
    if not hits:
        print(f'{keyword!r} not found. Try one of:', [w for w, _ in freq.most_common(10)])

kwic(tokens, 'cloak')

## 6. Collocations: what sits next to a word

Which words tend to appear beside a given word? That's a first, rough measure of meaning by association.

In [ ]:
def neighbours(tokens, keyword, width=2):
    near = Counter()
    for k, w in enumerate(tokens):
        if w == keyword:
            for j in range(max(0, k - width), min(len(tokens), k + width + 1)):
                if j != k:
                    near[tokens[j]] += 1
    return near.most_common(8)

print('Words near "wind":', neighbours(tokens, 'wind'))

## 7. Lemmatizing (the idea)

*blew*, *blow*, and *blowing* are three tokens but one **lemma**. A real lemmatizer knows the whole language; the version here is a small hand-built map, enough to show the move. A real corpus needs a proper tool.

In [ ]:
lemma_map = {
    'blew': 'blow', 'blows': 'blow', 'blowing': 'blow',
    'was': 'be', 'were': 'be', 'is': 'be',
}
lemmatized = [lemma_map.get(w, w) for w in tokens]

print('blow (all forms):', Counter(lemmatized)['blow'], 'occurrences after lemmatizing')
# === TO BUILD: replace lemma_map with a real lemmatizer for your language ===

## 8. Compare two texts

Most philological questions are comparative. The simplest comparison: which words appear in one text but not the other?

In [ ]:
text_b = 'the sun was warm and the traveler was glad to rest in the warm sun'.split()
set_a, set_b = set(tokens), set(text_b)

print('Only in the fable:', sorted(set_a - set_b)[:10], '...')
print('Only in text B:   ', sorted(set_b - set_a))
print('Shared:           ', sorted(set_a & set_b))

## 9. Link a word to a glossary

> 🔧 **TO BUILD**—connect a few words to real glossary entries. The link is just a mapping from a word to a URL.

In [ ]:
# === TO BUILD: map a few words to real glossary entries ===
glossary = {
    # 'word': 'https://zodiacglossary.github.io/entry/...',
}
for w in list(freq)[:5]:
    print(f'{w:10} ->', glossary.get(w, '(no entry yet)'))

## Your turn & project check-in
- Run the concordance on a different word. Anything unexpected in how it's used?
- **Check-in:** what's your corpus, and what counts as one unit in it—token, lemma, or sign?